# Practica M31: DataFrames, estadísticas, groupby y lambda
#### En este Notebook voy a trabajar con fifa_eda.csv para obtener estadísticas generales y por nacionalidad. Después añadiré dos jugadores inventados, compararé los resultados y crearé nuevas columnas con funciones lambda.

## Paso 1. Importar la librería
#### Utilizo pandas para leer y trabajar con los datos. No necesito instalar paquetes adicionales si ya tengo pandas en Anaconda.

In [ ]:
import os
import pandas as pd

## Paso 2. Entrar a Downloads y leer el CSV
#### Busco el archivo en mi carpeta de Descargas. Si está como fifa_eda(1).csv, también lo puedo utilizar.

In [ ]:
# Cambio el directorio a Downloads
os.chdir("/Users/santiago/Downloads")
print("Directorio:", os.getcwd())

# Reviso cuál nombre tiene mi archivo
if os.path.exists("fifa_eda.csv"):
    archivo = "fifa_eda.csv"
elif os.path.exists("fifa_eda(1).csv"):
    archivo = "fifa_eda(1).csv"
else:
    raise FileNotFoundError("No encontré fifa_eda.csv en Downloads")

fifa = pd.read_csv(archivo)
print("Archivo leído:", archivo)
print("Número de jugadores:", len(fifa))
print("Columnas:", len(fifa.columns))

## Paso 3. Visualizar una muestra de 20 jugadores
#### Con sample() selecciono 20 filas aleatorias. random_state permite obtener la misma muestra cada vez que ejecuto el código.

In [ ]:
print("Muestra de 20 jugadores:")
display(fifa.sample(20, random_state=42))

## Paso 4. Revisar y preparar los datos
#### Reutilizo la conversión numérica del ejercicio anterior. En este CSV, Overall, Potential y Age ya tienen valores numéricos, pero reviso sus tipos para asegurarme de poder calcular las estadísticas.

In [ ]:
# Reviso los tipos de las columnas que utilizaré
print(fifa[["Overall", "Potential", "Age", "Value", "Wage"]].dtypes)

# Convierto los datos a numéricos por seguridad
for columna in ["Overall", "Potential", "Age", "Value", "Wage"]:
    fifa[columna] = pd.to_numeric(fifa[columna], errors="coerce")

# Compruebo que todo esté listo
print("\nTipos después de convertir:")
print(fifa[["Overall", "Potential", "Age", "Value", "Wage"]].dtypes)

## Paso 5. Calcular media, mínimo y máximo (antes)
#### Calculo estas estadísticas para Overall, Potential y Age. Primero las obtengo para todos los jugadores y después las agrupo según su nacionalidad con groupby().

In [ ]:
# Estadísticas de todos los jugadores
columnas = ["Overall", "Potential", "Age"]
estadisticas_antes = fifa[columnas].agg(["mean", "min", "max"]).round(2)

print("Estadísticas generales ANTES:")
display(estadisticas_antes)

# Estadísticas por país
estadisticas_pais_antes = fifa.groupby("Nationality")[columnas].agg(["mean", "min", "max"]).round(2)

print("Estadísticas por país (primeros 20):")
display(estadisticas_pais_antes.head(20))
print("Total de nacionalidades:", len(estadisticas_pais_antes))

## Paso 6. Crear dos jugadores ficticios y agregarlos
#### Creo dos diccionarios con todas las columnas del archivo y los convierto en un DataFrame. Con pd.concat() los uno al dataset original, sin alterar el archivo CSV de origen.

In [ ]:
# Genero dos ID nuevos para no repetir los existentes
id_nuevo = int(fifa["ID"].max()) + 1

jugadores_ficticios = [
    {
        "ID": id_nuevo,
        "Name": "Alex Rivera (Ficticio)",
        "Age": 22,
        "Nationality": "Mexico",
        "Overall": 82,
        "Potential": 92,
        "Club": "Club Ejemplo MX",
        "Value": 25000,
        "Wage": 80,
        "Preferred Foot": "Right",
        "International Reputation": 2,
        "Skill Moves": 4,
        "Position": "ST",
        "Joined": 2025,
        "Contract Valid Until": "2029-01-01",
        "Height": 5.83,
        "Weight": 165,
        "Release Clause": 35000
    },
    {
        "ID": id_nuevo + 1,
        "Name": "Sam Taylor (Ficticio)",
        "Age": 24,
        "Nationality": "Australia",
        "Overall": 79,
        "Potential": 88,
        "Club": "Club Ejemplo AU",
        "Value": 18000,
        "Wage": 65,
        "Preferred Foot": "Left",
        "International Reputation": 2,
        "Skill Moves": 3,
        "Position": "CM",
        "Joined": 2024,
        "Contract Valid Until": "2028-01-01",
        "Height": 6.00,
        "Weight": 172,
        "Release Clause": 29000
    }
]

# Los convierto en un DataFrame con las mismas 18 columnas
nuevos = pd.DataFrame(jugadores_ficticios, columns=fifa.columns)

# Agrego los dos jugadores al final
datos_actualizados = pd.concat([fifa, nuevos], ignore_index=True)

print("Total original:", len(fifa))
print("Total con los 2 ficticios:", len(datos_actualizados))
print("Últimas 5 filas:")
display(datos_actualizados.tail())

## Paso 7. Calcular nuevamente las estadísticas y compararlas
#### Vuelvo a calcular media, mínimo y máximo después de agregar los dos jugadores. Comparo los valores generales y los de México y Australia, que son las nacionalidades de mis jugadores ficticios.

In [ ]:
# Nuevas estadísticas generales
estadisticas_despues = datos_actualizados[columnas].agg(["mean", "min", "max"]).round(2)

print("Estadísticas generales DESPUÉS:")
display(estadisticas_despues)

# Pongo los resultados uno al lado del otro
comparacion_general = pd.concat(
    [estadisticas_antes, estadisticas_despues],
    axis=1,
    keys=["Antes", "Después"]
)
print("Comparación ANTES y DESPUÉS:")
display(comparacion_general)

# Vuelvo a agrupar por nacionalidad
estadisticas_pais_despues = datos_actualizados.groupby("Nationality")[columnas].agg(["mean", "min", "max"]).round(2)

print("México y Australia ANTES:")
display(estadisticas_pais_antes.loc[["Mexico", "Australia"]])
print("México y Australia DESPUÉS:")
display(estadisticas_pais_despues.loc[["Mexico", "Australia"]])

## Paso 8. Usar lambda para agregar capital y zona horaria
#### Primero preparo un diccionario de consulta con las 164 nacionalidades que aparecen en el CSV. Cada país tiene una capital y una zona horaria de referencia (la de su capital). Algunas naciones tienen varios husos horarios, por lo que una sola zona no describe todo el país.
#### Después uso apply() y dos funciones lambda para crear las columnas Capital y Zona horaria. El diccionario es solo información de apoyo; lo que practico en este paso es el uso de lambda.

In [ ]:
# Capital y zona horaria de referencia por nacionalidad
datos_paises = {
    'Afghanistan': ('Kabul', 'Asia/Kabul'),
    'Albania': ('Tirana', 'Europe/Tirane'),
    'Algeria': ('Algiers', 'Africa/Algiers'),
    'Andorra': ('Andorra la Vieja', 'Europe/Andorra'),
    'Angola': ('Luanda', 'Africa/Luanda'),
    'Antigua & Barbuda': ('Saint John’s', 'America/Antigua'),
    'Argentina': ('Buenos Aires', 'America/Argentina/Buenos_Aires'),
    'Armenia': ('Yerevan', 'Asia/Yerevan'),
    'Australia': ('Canberra', 'Australia/Sydney'),
    'Austria': ('Vienna', 'Europe/Vienna'),
    'Azerbaijan': ('Baku', 'Asia/Baku'),
    'Barbados': ('Bridgetown', 'America/Barbados'),
    'Belarus': ('Minsk', 'Europe/Minsk'),
    'Belgium': ('Brussels', 'Europe/Brussels'),
    'Belize': ('Belmopan', 'America/Belize'),
    'Benin': ('Porto-Novo', 'Africa/Porto-Novo'),
    'Bermuda': ('Hamilton', 'Atlantic/Bermuda'),
    'Bolivia': ('Sucre', 'America/La_Paz'),
    'Bosnia Herzegovina': ('Sarajevo', 'Europe/Sarajevo'),
    'Botswana': ('Gaborone', 'Africa/Gaborone'),
    'Brazil': ('Brasilia', 'America/Sao_Paulo'),
    'Bulgaria': ('Sofia', 'Europe/Sofia'),
    'Burkina Faso': ('Ouagadougou', 'Africa/Ouagadougou'),
    'Burundi': ('Bujumbura', 'Africa/Bujumbura'),
    'Cameroon': ('Yaoundé', 'Africa/Douala'),
    'Canada': ('Ottawa', 'America/Toronto'),
    'Cape Verde': ('Praia', 'Atlantic/Cape_Verde'),
    'Central African Rep.': ('Bangui', 'Africa/Bangui'),
    'Chad': ("N'Djamena", 'Africa/Ndjamena'),
    'Chile': ('Santiago', 'America/Santiago'),
    'China PR': ('Pekín', 'Asia/Shanghai'),
    'Colombia': ('Bogotá', 'America/Bogota'),
    'Comoros': ('Moroni', 'Indian/Comoro'),
    'Congo': ('Brazzaville', 'Africa/Brazzaville'),
    'Costa Rica': ('San José', 'America/Costa_Rica'),
    'Croatia': ('Zagreb', 'Europe/Zagreb'),
    'Cuba': ('Havana', 'America/Havana'),
    'Curacao': ('Willemstad', 'America/Curacao'),
    'Cyprus': ('Nicosia', 'Asia/Nicosia'),
    'Czech Republic': ('Prague', 'Europe/Prague'),
    'DR Congo': ('Kinshasa', 'Africa/Kinshasa'),
    'Denmark': ('Copenhagen', 'Europe/Copenhagen'),
    'Dominican Republic': ('Santo Domingo', 'America/Santo_Domingo'),
    'Ecuador': ('Quito', 'America/Guayaquil'),
    'Egypt': ('Cairo', 'Africa/Cairo'),
    'El Salvador': ('San Salvador', 'America/El_Salvador'),
    'England': ('Londres', 'Europe/London'),
    'Equatorial Guinea': ('Malabo', 'Africa/Malabo'),
    'Eritrea': ('Asmara', 'Africa/Asmara'),
    'Estonia': ('Tallinn', 'Europe/Tallinn'),
    'Ethiopia': ('Addis Ababa', 'Africa/Addis_Ababa'),
    'FYR Macedonia': ('Skopie', 'Europe/Skopje'),
    'Faroe Islands': ('Tórshavn', 'Atlantic/Faroe'),
    'Fiji': ('Suva', 'Pacific/Fiji'),
    'Finland': ('Helsinki', 'Europe/Helsinki'),
    'France': ('Paris', 'Europe/Paris'),
    'Gabon': ('Libreville', 'Africa/Libreville'),
    'Gambia': ('Banjul', 'Africa/Banjul'),
    'Georgia': ('Tiflis', 'Asia/Tbilisi'),
    'Germany': ('Berlin', 'Europe/Berlin'),
    'Ghana': ('Accra', 'Africa/Accra'),
    'Greece': ('Athens', 'Europe/Athens'),
    'Grenada': ("St. George's", 'America/Grenada'),
    'Guam': ('Hagåtña', 'Pacific/Guam'),
    'Guatemala': ('Guatemala City', 'America/Guatemala'),
    'Guinea': ('Conakry', 'Africa/Conakry'),
    'Guinea Bissau': ('Bisáu', 'Africa/Bissau'),
    'Guyana': ('Georgetown', 'America/Guyana'),
    'Haiti': ('Port-au-Prince', 'America/Port-au-Prince'),
    'Honduras': ('Tegucigalpa', 'America/Tegucigalpa'),
    'Hong Kong': ('City of Victoria', 'Asia/Hong_Kong'),
    'Hungary': ('Budapest', 'Europe/Budapest'),
    'Iceland': ('Reykjavik', 'Atlantic/Reykjavik'),
    'India': ('New Delhi', 'Asia/Kolkata'),
    'Indonesia': ('Yakarta', 'Asia/Jakarta'),
    'Iran': ('Tehran', 'Asia/Tehran'),
    'Iraq': ('Baghdad', 'Asia/Baghdad'),
    'Israel': ('Jerusalem', 'Asia/Jerusalem'),
    'Italy': ('Rome', 'Europe/Rome'),
    'Ivory Coast': ('Yamusukro', 'Africa/Abidjan'),
    'Jamaica': ('Kingston', 'America/Jamaica'),
    'Japan': ('Tokyo', 'Asia/Tokyo'),
    'Jordan': ('Amman', 'Asia/Amman'),
    'Kazakhstan': ('Astaná', 'Asia/Almaty'),
    'Kenya': ('Nairobi', 'Africa/Nairobi'),
    'Korea DPR': ('Pionyang', 'Asia/Pyongyang'),
    'Korea Republic': ('Seúl', 'Asia/Seoul'),
    'Kosovo': ('Pristina', 'Europe/Belgrade'),
    'Kuwait': ('Kuwait City', 'Asia/Kuwait'),
    'Latvia': ('Riga', 'Europe/Riga'),
    'Lebanon': ('Beirut', 'Asia/Beirut'),
    'Liberia': ('Monrovia', 'Africa/Monrovia'),
    'Libya': ('Tripoli', 'Africa/Tripoli'),
    'Liechtenstein': ('Vaduz', 'Europe/Vaduz'),
    'Lithuania': ('Vilnius', 'Europe/Vilnius'),
    'Luxembourg': ('Luxembourg', 'Europe/Luxembourg'),
    'Madagascar': ('Antananarivo', 'Indian/Antananarivo'),
    'Mali': ('Bamako', 'Africa/Bamako'),
    'Malta': ('Valletta', 'Europe/Malta'),
    'Mauritania': ('Nouakchott', 'Africa/Nouakchott'),
    'Mauritius': ('Port Louis', 'Indian/Mauritius'),
    'Mexico': ('Ciudad de México', 'America/Mexico_City'),
    'Moldova': ('Chișinău', 'Europe/Chisinau'),
    'Montenegro': ('Podgorica', 'Europe/Podgorica'),
    'Montserrat': ('Brades (sede de gobierno)', 'America/Montserrat'),
    'Morocco': ('Rabat', 'Africa/Casablanca'),
    'Mozambique': ('Maputo', 'Africa/Maputo'),
    'Namibia': ('Windhoek', 'Africa/Windhoek'),
    'Netherlands': ('Ámsterdam', 'Europe/Amsterdam'),
    'New Caledonia': ('Numea', 'Pacific/Noumea'),
    'New Zealand': ('Wellington', 'Pacific/Auckland'),
    'Nicaragua': ('Managua', 'America/Managua'),
    'Niger': ('Niamey', 'Africa/Niamey'),
    'Nigeria': ('Abuja', 'Africa/Lagos'),
    'Northern Ireland': ('Belfast', 'Europe/London'),
    'Norway': ('Oslo', 'Europe/Oslo'),
    'Oman': ('Muscat', 'Asia/Muscat'),
    'Palestine': ('Ramala (sede administrativa)', 'Asia/Hebron'),
    'Panama': ('Panama City', 'America/Panama'),
    'Paraguay': ('Asunción', 'America/Asuncion'),
    'Peru': ('Lima', 'America/Lima'),
    'Philippines': ('Manila', 'Asia/Manila'),
    'Poland': ('Warsaw', 'Europe/Warsaw'),
    'Portugal': ('Lisboa', 'Europe/Lisbon'),
    'Puerto Rico': ('San Juan', 'America/Puerto_Rico'),
    'Qatar': ('Doha', 'Asia/Qatar'),
    'Republic of Ireland': ('Dublín', 'Europe/Dublin'),
    'Romania': ('Bucharest', 'Europe/Bucharest'),
    'Russia': ('Moscú', 'Europe/Moscow'),
    'Rwanda': ('Kigali', 'Africa/Kigali'),
    'Saudi Arabia': ('Riyadh', 'Asia/Riyadh'),
    'Scotland': ('Edimburgo', 'Europe/London'),
    'Senegal': ('Dakar', 'Africa/Dakar'),
    'Serbia': ('Belgrade', 'Europe/Belgrade'),
    'Sierra Leone': ('Freetown', 'Africa/Freetown'),
    'Slovakia': ('Bratislava', 'Europe/Bratislava'),
    'Slovenia': ('Ljubljana', 'Europe/Ljubljana'),
    'South Africa': ('Pretoria (administrativa)', 'Africa/Johannesburg'),
    'South Sudan': ('Juba', 'Africa/Juba'),
    'Spain': ('Madrid', 'Europe/Madrid'),
    'St Kitts Nevis': ('Basseterre', 'America/St_Kitts'),
    'St Lucia': ('Castries', 'America/St_Lucia'),
    'Sudan': ('Khartoum', 'Africa/Khartoum'),
    'Suriname': ('Paramaribo', 'America/Paramaribo'),
    'Sweden': ('Stockholm', 'Europe/Stockholm'),
    'Switzerland': ('Bern', 'Europe/Zurich'),
    'Syria': ('Damascus', 'Asia/Damascus'),
    'São Tomé & Príncipe': ('Santo Tomé', 'Africa/Sao_Tome'),
    'Tanzania': ('Dodoma', 'Africa/Dar_es_Salaam'),
    'Thailand': ('Bangkok', 'Asia/Bangkok'),
    'Togo': ('Lomé', 'Africa/Lome'),
    'Trinidad & Tobago': ('Puerto España', 'America/Port_of_Spain'),
    'Tunisia': ('Tunis', 'Africa/Tunis'),
    'Turkey': ('Ankara', 'Europe/Istanbul'),
    'Uganda': ('Kampala', 'Africa/Kampala'),
    'Ukraine': ('Kiev', 'Europe/Kyiv'),
    'United Arab Emirates': ('Abu Dhabi', 'Asia/Dubai'),
    'United States': ('Washington D. C.', 'America/New_York'),
    'Uruguay': ('Montevideo', 'America/Montevideo'),
    'Uzbekistan': ('Taskent', 'Asia/Tashkent'),
    'Venezuela': ('Caracas', 'America/Caracas'),
    'Wales': ('Cardiff', 'Europe/London'),
    'Zambia': ('Lusaka', 'Africa/Lusaka'),
    'Zimbabwe': ('Harare', 'Africa/Harare'),
}

In [ ]:
# Creo las dos columnas utilizando lambda
# get() devuelve una referencia si algún país no aparece en el diccionario
datos_actualizados["Capital"] = datos_actualizados["Nationality"].apply(
    lambda pais: datos_paises.get(pais, ("No disponible", "No disponible"))[0]
)

datos_actualizados["Zona horaria"] = datos_actualizados["Nationality"].apply(
    lambda pais: datos_paises.get(pais, ("No disponible", "No disponible"))[1]
)

print("Ejemplos de capital y zona horaria:")
display(datos_actualizados[["Name", "Nationality", "Capital", "Zona horaria"]].head(15))
print("Países sin referencia:", (datos_actualizados["Capital"] == "No disponible").sum())

## Paso 9. Usar lambda para identificar candidatos al Balón de Oro
#### Si el Potential es mayor a 85, escribo Candidato. En los demás casos escribo Next Year. Este criterio es solo el del ejercicio y no es una predicción real del premio.

In [ ]:
# Clasifico a los jugadores según su potencial
datos_actualizados["Balón de Oro"] = datos_actualizados["Potential"].apply(
    lambda potencial: "Candidato" if potencial > 85 else "Next Year"
)

print("Cantidad de jugadores por clasificación:")
print(datos_actualizados["Balón de Oro"].value_counts())

print("Ejemplos de candidatos:")
candidatos = datos_actualizados[datos_actualizados["Balón de Oro"] == "Candidato"]
display(candidatos[["Name", "Nationality", "Potential", "Balón de Oro"]].head(20))

print("Mis dos jugadores ficticios:")
display(datos_actualizados[["Name", "Potential", "Capital", "Zona horaria", "Balón de Oro"]].tail(2))

## Paso 10. Guardar los resultados
#### Guardo una copia actualizada en Downloads, además de las estadísticas por nacionalidad. El CSV original permanece sin cambios.

In [ ]:
# Guardo el DataFrame con los nuevos jugadores y columnas
datos_actualizados.to_csv("fifa_dataframe_lambdas.csv", index=False)

# Guardo también las estadísticas por nacionalidad
estadisticas_pais_despues.to_csv("fifa_estadisticas_por_pais.csv")

print("Archivos creados en Downloads:")
print("- fifa_dataframe_lambdas.csv")
print("- fifa_estadisticas_por_pais.csv")